# App rápida (sin reentrenar)
Descarga los resultados ya calculados desde el **Release** del repositorio (`resultados_para_colab.zip`, con el contenido de `data/processed`) y levanta la app.

*Entorno de ejecución → Ejecutar todas* y abre la liga `https://….trycloudflare.com` que se imprime al final.

In [ ]:
%cd /content
import os
if not os.path.isdir('ParticipacionCiudadanaINE'):
    !git clone -q https://github.com/AndreaHortaHernandez/ParticipacionCiudadanaINE.git
%cd /content/ParticipacionCiudadanaINE
!git pull -q
!pip -q install -r requirements.txt

### Resultados precalculados
Se descargan del último Release. Para usar una versión específica, cambia `latest/download` por `download/v1.0` (o el tag que corresponda).

In [ ]:
import os, glob, shutil, zipfile, urllib.request

URL = 'https://github.com/AndreaHortaHernandez/ParticipacionCiudadanaINE/releases/latest/download/resultados_para_colab.zip'
urllib.request.urlretrieve(URL, 'resultados_para_colab.zip')
zipfile.ZipFile('resultados_para_colab.zip').extractall('tmp_zip')

# Acepta el zip tanto con los archivos en la raíz como dentro de data/processed/
encontrados = glob.glob('tmp_zip/**/riesgo_*.parquet', recursive=True)
assert encontrados, 'El zip no trae ningún riesgo_*.parquet'
os.makedirs('data/processed', exist_ok=True)
shutil.copytree(os.path.dirname(encontrados[0]), 'data/processed', dirs_exist_ok=True)
shutil.rmtree('tmp_zip'); os.remove('resultados_para_colab.zip')
print(sorted(os.listdir('data/processed')))

### Cartografía y agente
Si el Release trae `geo.zip` (mapas ya procesados), se usa directo y este paso tarda segundos. Si no, descarga los shapefiles del INE desde el Release (el sitio `pautas.ine.mx` ya no existe) y los procesa con `src.geo` (~5 min). Si la cartografía falla, la app funciona igual, pero sin mapas.

In [ ]:
%%bash
REL=https://github.com/AndreaHortaHernandez/ParticipacionCiudadanaINE/releases/latest/download
if [ ! -f data/processed/geo/entidades.geojson ] && curl -fsSL -o geo.zip "$REL/geo.zip"; then
  unzip -oq geo.zip -d data/processed && echo 'Cartografía ya procesada (geo.zip del Release).'
  rm -f geo.zip
fi
if [ -f data/processed/geo/entidades.geojson ]; then
  echo 'Cartografía incluida en el zip: se omite.'
else
  mkdir -p data/geo && cd data/geo
  for f in SECCION MUNICIPIO ENTIDAD; do
    if curl -fsSL -o $f.zip "$REL/$f.zip"; then
      mkdir -p $f && unzip -oq $f.zip -d $f
    else
      echo "⚠️ No se pudo descargar $f.zip del Release (¿ya se subió?)"
    fi
    rm -f $f.zip
  done
  cd ../..
  python -W ignore -m src.geo || echo '⚠️ No se pudo generar la cartografía: la app correrá sin mapas.'
fi
if [ -f data/processed/intervenciones_2027.parquet ]; then
  echo 'Plan del agente incluido en el zip: se omite.'
else
  python -m src.agent --anio 2027 --presupuesto-pct 10
fi

### App
Abre la liga `https://….trycloudflare.com` que se imprime. Funciona mientras esta sesión siga abierta.

In [ ]:
%%bash
wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared && chmod +x cloudflared
nohup streamlit run app.py --server.port 8501 --server.headless true > streamlit.log 2>&1 &
nohup ./cloudflared tunnel --url http://localhost:8501 > cloudflared.log 2>&1 &
# Espera hasta 60 s a que aparezca la liga
for i in $(seq 1 30); do
  URL=$(grep -oE 'https://[a-z0-9-]+\.trycloudflare\.com' cloudflared.log | head -1)
  [ -n "$URL" ] && break; sleep 2
done
echo "${URL:-No apareció la liga; revisa cloudflared.log}"

### (Opcional, una sola vez) Guardar los mapas procesados
Si el Release todavía no tiene `geo.zip`, corre esta celda después de que se generó la cartografía, descarga el archivo y súbelo al Release (*Releases → v1.0 → Edit → arrastrar*). Desde entonces este notebook se salta `src.geo`.

In [ ]:
!cd data/processed && zip -qr /content/geo.zip geo && ls -lh /content/geo.zip
from google.colab import files
files.download('/content/geo.zip')